# Unit Testing Assignment

**Task:** Implement unit tests, integration tests, and end-to-end tests for a sample
application using frameworks like JUnit, TestNG, or Selenium.

**Sample application:** A simple Java `Calculator` class + a tiny static HTML page, so we can
demonstrate all three test levels:
- **Unit tests** → JUnit 5, testing `Calculator` in isolation
- **Integration tests** → TestNG, testing `Calculator` + a `CalculatorService` wrapper together
- **End-to-end tests** → Selenium WebDriver, testing the HTML calculator page in a real browser


## 1. Sample Application

In [1]:
import os
os.makedirs('project/src/main/java/com/example', exist_ok=True)

with open('project/src/main/java/com/example/Calculator.java', 'w') as f:
    f.write('''package com.example;

public class Calculator {
    public int add(int a, int b) { return a + b; }
    public int subtract(int a, int b) { return a - b; }
    public int multiply(int a, int b) { return a * b; }
    public double divide(int a, int b) {
        if (b == 0) throw new ArithmeticException("Division by zero");
        return (double) a / b;
    }
}
''')

with open('project/src/main/java/com/example/CalculatorService.java', 'w') as f:
    f.write('''package com.example;

// A thin service layer wrapping Calculator, used to demonstrate integration tests
// between two collaborating components.
public class CalculatorService {
    private final Calculator calculator = new Calculator();

    public String computeSummary(int a, int b) {
        return String.format("add=%d, subtract=%d, multiply=%d, divide=%.2f",
                calculator.add(a, b),
                calculator.subtract(a, b),
                calculator.multiply(a, b),
                calculator.divide(a, b));
    }
}
''')
print("Created Calculator.java and CalculatorService.java")


Created Calculator.java and CalculatorService.java


In [2]:
import os
os.makedirs('project/web', exist_ok=True)
with open('project/web/index.html', 'w') as f:
    f.write('''<!DOCTYPE html>
<html>
<head><title>Simple Calculator</title></head>
<body>
  <input id="a" type="number" value="0">
  <input id="b" type="number" value="0">
  <button id="addBtn" onclick="document.getElementById('result').innerText =
      Number(document.getElementById('a').value) + Number(document.getElementById('b').value)">Add</button>
  <p id="result">0</p>
</body>
</html>
''')
print("Created project/web/index.html (used by the Selenium E2E test)")


Created project/web/index.html (used by the Selenium E2E test)


## 2. Unit Tests — JUnit 5

In [3]:
import os
os.makedirs('project/src/test/java/com/example', exist_ok=True)
with open('project/src/test/java/com/example/CalculatorTest.java', 'w') as f:
    f.write('''package com.example;

import org.junit.jupiter.api.Test;
import static org.junit.jupiter.api.Assertions.*;

class CalculatorTest {

    private final Calculator calculator = new Calculator();

    @Test
    void testAdd() {
        assertEquals(5, calculator.add(2, 3));
    }

    @Test
    void testSubtract() {
        assertEquals(-1, calculator.subtract(2, 3));
    }

    @Test
    void testMultiply() {
        assertEquals(6, calculator.multiply(2, 3));
    }

    @Test
    void testDivide() {
        assertEquals(2.5, calculator.divide(5, 2), 0.001);
    }

    @Test
    void testDivideByZeroThrows() {
        assertThrows(ArithmeticException.class, () -> calculator.divide(5, 0));
    }
}
''')
print("Created CalculatorTest.java (JUnit 5)")


Created CalculatorTest.java (JUnit 5)


## 3. Integration Tests — TestNG

In [4]:
with open('project/src/test/java/com/example/CalculatorServiceIntegrationTest.java', 'w') as f:
    f.write('''package com.example;

import org.testng.annotations.Test;
import static org.testng.Assert.*;

// Validates that CalculatorService correctly integrates with Calculator
public class CalculatorServiceIntegrationTest {

    private final CalculatorService service = new CalculatorService();

    @Test
    public void testComputeSummary() {
        String summary = service.computeSummary(10, 5);
        assertTrue(summary.contains("add=15"));
        assertTrue(summary.contains("subtract=5"));
        assertTrue(summary.contains("multiply=50"));
        assertTrue(summary.contains("divide=2.00"));
    }

    @Test(expectedExceptions = ArithmeticException.class)
    public void testComputeSummaryDivideByZero() {
        service.computeSummary(10, 0);
    }
}
''')
print("Created CalculatorServiceIntegrationTest.java (TestNG)")


Created CalculatorServiceIntegrationTest.java (TestNG)


## 4. End-to-End Tests — Selenium WebDriver

In [5]:
with open('project/src/test/java/com/example/CalculatorE2ETest.java', 'w') as f:
    f.write('''package com.example;

import org.junit.jupiter.api.*;
import org.openqa.selenium.*;
import org.openqa.selenium.chrome.ChromeDriver;

import java.io.File;

// Drives the actual HTML page (web/index.html) in a real browser end-to-end
class CalculatorE2ETest {

    private WebDriver driver;

    @BeforeEach
    void setUp() {
        // System.setProperty("webdriver.chrome.driver", "/path/to/chromedriver"); // Selenium Manager configures this automatically
        driver = new ChromeDriver();
        File page = new File("web/index.html");
        driver.get("file://" + page.getAbsolutePath());
    }

    @Test
    void testAddViaUI() {
        driver.findElement(By.id("a")).clear();
        driver.findElement(By.id("a")).sendKeys("4");
        driver.findElement(By.id("b")).clear();
        driver.findElement(By.id("b")).sendKeys("6");
        driver.findElement(By.id("addBtn")).click();

        String result = driver.findElement(By.id("result")).getText();
        Assertions.assertEquals("10", result);
    }

    @AfterEach
    void tearDown() {
        if (driver != null) driver.quit();
    }
}
''')
print("Created CalculatorE2ETest.java (Selenium)")


Created CalculatorE2ETest.java (Selenium)


## 5. Maven `pom.xml` (dependencies + running all three suites)

In [6]:
with open('project/pom.xml', 'w') as f:
    f.write('''<project xmlns="http://maven.apache.org/POM/4.0.0">
  <modelVersion>4.0.0</modelVersion>
  <groupId>com.example</groupId>
  <artifactId>calculator-tests</artifactId>
  <version>1.0.0</version>
  <properties>
    <maven.compiler.source>17</maven.compiler.source>
    <maven.compiler.target>17</maven.compiler.target>
  </properties>
  <dependencies>
    <dependency>
      <groupId>org.junit.jupiter</groupId>
      <artifactId>junit-jupiter</artifactId>
      <version>5.10.0</version>
      <scope>test</scope>
    </dependency>
    <dependency>
      <groupId>org.testng</groupId>
      <artifactId>testng</artifactId>
      <version>7.9.0</version>
      <scope>test</scope>
    </dependency>
    <dependency>
      <groupId>org.seleniumhq.selenium</groupId>
      <artifactId>selenium-java</artifactId>
      <version>4.19.1</version>
      <scope>test</scope>
    </dependency>
  </dependencies>
  <build>
    <plugins>
      <plugin>
        <groupId>org.apache.maven.plugins</groupId>
        <artifactId>maven-surefire-plugin</artifactId>
        <version>3.2.5</version>
      </plugin>
    </plugins>
  </build>
</project>
''')
print("Created pom.xml — run all tests with: mvn test")


Created pom.xml — run all tests with: mvn test
